In [ ]:
import os, gc, torch, numpy as np, pandas as pd
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# ==================== PATH SETTINGS ====================
chunk_dir = "/kaggle/input/datasets/abammar/autoencoder/AutoEncoder/Chunk"
output_dir = "/kaggle/working/latent_features_output"
model_dir = "/kaggle/working/saved_models"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# ==================== CONFIGURATION ====================
latent_dim = 150
epochs = 15
batch_size = 16
start_chunk = 1
end_chunk = 7
device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    print(f"Using {gpu_count} GPU(s): {gpu_names}")
else:
    print("No GPU detected, running on CPU")

# ==================== MODEL DEFINITION ====================
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 512), nn.ReLU(),
            nn.Linear(512, input_dim), nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z

# ==================== TRAINING LOOP ====================
best_loss_overall = float("inf")
global_best_path = os.path.join(model_dir, "best_encoder_150latent.pth")

for ch in range(start_chunk, end_chunk + 1):
    npz_path = os.path.join(chunk_dir, f"chunk_{ch}.npz")
    if not os.path.exists(npz_path):
        print(f"Chunk {ch} not found, skipping...")
        continue

    print(f"\nLoading Chunk {ch}...")
    data = np.load(npz_path, allow_pickle=True)
    X = data["X"]
    sample_ids = data["sample_ids"]
    print(f"Loaded: {X.shape[0]} samples x {X.shape[1]} CpGs")

    # ---- Normalize ----
    col_means = np.nanmean(X, axis=0)
    col_means = np.nan_to_num(col_means, nan=0.5)   # fix empty slice warning
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])
    X = np.nan_to_num(X, nan=0.5)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    del X, X_scaled
    gc.collect()

    # ---- Build Model ----
    input_dim = X_tensor.shape[1]
    model = Autoencoder(input_dim, latent_dim).to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=True)

    # ---- Train ----
    best_chunk_loss = float("inf")
    print(f"Training Autoencoder on Chunk {ch}...")

    for ep in range(epochs):
        model.train()
        total_loss = 0
        for (batch_x,) in loader:
            batch_x = batch_x.to(device)
            optimizer.zero_grad()
            xhat, _ = model(batch_x)
            loss = criterion(xhat, batch_x)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        print(f"   Epoch {ep+1}/{epochs} | Loss: {avg_loss:.6f}")

        # ---- Save Best Model for This Chunk ----
        if avg_loss < best_chunk_loss:
            best_chunk_loss = avg_loss
            chunk_model_path = os.path.join(model_dir, f"best_encoder_chunk{ch}.pth")
            raw_model = model.module if hasattr(model, "module") else model
            torch.save({
                "epoch": ep + 1,
                "input_dim": input_dim,
                "latent_dim": latent_dim,
                "model_state_dict": raw_model.state_dict(),
                "loss": best_chunk_loss,
            }, chunk_model_path)

    # ---- Save Global Best Model ----
    if best_chunk_loss < best_loss_overall:
        best_loss_overall = best_chunk_loss
        raw_model = model.module if hasattr(model, "module") else model
        torch.save({
            "input_dim": input_dim,
            "latent_dim": latent_dim,
            "model_state_dict": raw_model.state_dict(),
            "loss": best_loss_overall,
        }, global_best_path)
        print(f"   New GLOBAL best model saved! Loss={best_loss_overall:.6f}")

    # ---- Extract Latent Features (FIXED) ----
    raw_model = model.module if hasattr(model, "module") else model
    raw_model.eval()
    with torch.no_grad():
        latent_features = []
        for (batch_x,) in DataLoader(TensorDataset(X_tensor), batch_size=batch_size):
            batch_x = batch_x.to(device)
            z = raw_model.encoder(batch_x)   # FIX: encoder returns z directly
            latent_features.append(z.cpu().numpy())
        latent_matrix = np.vstack(latent_features)

    # ---- Save Latent CSV ----
    latent_df = pd.DataFrame(
        latent_matrix,
        columns=[f"latent_{i+1}" for i in range(latent_dim)]
    )
    latent_df.insert(0, "sample_id", sample_ids)
    out_csv = os.path.join(output_dir, f"latent_chunk{ch}_{latent_dim}f.csv")
    latent_df.to_csv(out_csv, index=False)
    print(f"Saved latent features -> {out_csv}")

    # ---- Cleanup ----
    del model, optimizer, criterion, loader, latent_df, latent_features, latent_matrix
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Memory cleared after chunk {ch}")

print(f"\nAll chunks done! Global best model loss: {best_loss_overall:.6f}")
print(f"Best model saved at: {global_best_path}")


In [ ]:
import os, gc, torch, numpy as np, pandas as pd
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# ==================== PATHS ====================
chunk_dir = "/kaggle/input/datasets/abammar/autoencoder/AutoEncoder/Chunk"
output_dir = "/kaggle/working/latent_features_output_final"
model_path = "/kaggle/working/saved_models/best_encoder_150latent.pth"
os.makedirs(output_dir, exist_ok=True)

latent_dim = 150
batch_size = 16
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==================== MODEL DEFINITION ====================
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 512), nn.ReLU(),
            nn.Linear(512, input_dim), nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z

# ==================== LOAD GLOBAL BEST MODEL ====================
print("Loading global best model...")
checkpoint = torch.load(model_path, map_location=device)
input_dim = checkpoint["input_dim"]

model = Autoencoder(input_dim, latent_dim).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Model loaded! input_dim={input_dim}, latent_dim={latent_dim}, best_loss={checkpoint['loss']:.6f}")

# ==================== EXTRACT ALL CHUNKS ====================
for ch in range(1, 8):
    npz_path = os.path.join(chunk_dir, f"chunk_{ch}.npz")
    if not os.path.exists(npz_path):
        print(f"Chunk {ch} not found, skipping...")
        continue

    print(f"\nProcessing Chunk {ch}...")
    data = np.load(npz_path, allow_pickle=True)
    X = data["X"]
    sample_ids = data["sample_ids"]

    # ---- Normalize (same as training) ----
    col_means = np.nanmean(X, axis=0)
    col_means = np.nan_to_num(col_means, nan=0.5)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])
    X = np.nan_to_num(X, nan=0.5)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    del X, X_scaled
    gc.collect()

    # ---- Extract Latent Features ----
    with torch.no_grad():
        latent_features = []
        for (batch_x,) in DataLoader(TensorDataset(X_tensor), batch_size=batch_size):
            batch_x = batch_x.to(device)
            z = model.encoder(batch_x)
            latent_features.append(z.cpu().numpy())
        latent_matrix = np.vstack(latent_features)

    # ---- Save CSV ----
    latent_df = pd.DataFrame(
        latent_matrix,
        columns=[f"latent_{i+1}" for i in range(latent_dim)]
    )
    latent_df.insert(0, "sample_id", sample_ids)
    out_csv = os.path.join(output_dir, f"latent_chunk{ch}_final.csv")
    latent_df.to_csv(out_csv, index=False)
    print(f"Saved -> {out_csv} ({len(sample_ids)} samples)")

    del X_tensor, latent_features, latent_matrix, latent_df
    gc.collect()

print("\nAll chunks re-extracted using the SAME global best model!")

# ==================== MERGE ALL ====================
print("\nMerging all chunks...")
all_chunks = []
for ch in range(1, 8):
    csv_path = os.path.join(output_dir, f"latent_chunk{ch}_final.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        all_chunks.append(df)
        print(f"  Chunk {ch}: {df.shape[0]} samples")

merged_df = pd.concat(all_chunks, ignore_index=True)
merged_path = os.path.join(output_dir, "ALL_latent_features_150f_FINAL.csv")
merged_df.to_csv(merged_path, index=False)
print(f"\nTotal samples merged: {merged_df.shape[0]}")
print(f"Final merged CSV saved -> {merged_path}")
